conda install pytorch torchvision torchaudio pytorch-cuda=12.4 -c pytorch -c nvidia

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 간단 실행 테스트
x = torch.randn(3, 3).to("cuda" if torch.cuda.is_available() else "cpu")
print("tensor device:", x.device)

In [ ]:
# ==================================================
# 0️⃣ 필수 라이브러리
# ==================================================
import os, math, time, base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from jinja2 import Template

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import shap

In [123]:
# -*- coding: utf-8 -*-
"""
🌱 완전 통합 파이프라인 (Plant → Region)
- CNN-BiLSTM 기반
- 1차 학습 / 조건부 2차 학습 (weight step)
- Region별 Fine-tuning
- Resume, AMP, EarlyStopping
- Permutation Importance 안전 분석 (안전하게 시퀀스 처리)
- HTML/CSV 보고서 자동 생성 + Plant vs Region 성능 비교
- 출력 강화: ETA, Best 표시, 상세 지표
- 음수 R² -> 0 보정 및 제외된 발전소 별도 표기
"""
import os, time, warnings, base64, webbrowser, traceback
import io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from jinja2 import Template
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ----------------------------
# === 사용자 설정 ===
# ----------------------------
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"
SAVE_DIR  = r"C:\ESG_Project1\cnn_lstm\output"
CKPT_DIR  = os.path.join(SAVE_DIR,"checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

TIME_COL, GROUP_COL, REGION_COL, TARGET_COL = "일시", "발전구분", "지역", "합산발전량(MWh)"
WEATHER_COLS = [
    "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
    "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"
]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

SEQ_LEN, HORIZON = 168, 24
BATCH = 128
EPOCHS = 50
FINE_TUNE_EPOCHS = 10
LR = 1e-3
PATIENCE = 0
PERM_SAMPLE = 50
RESUME = True   # Resume 기능 사용 여부 (전체 파이프라인 재개 가능)

# ----------------------------
# === 데이터 로드 + 전처리 ===
# ----------------------------
def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"] = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"] = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"] = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lag_diff(df, lags=[1,3,6,24]):
    df = df.copy()
    for lag in lags:
        df[f"lag_{lag}"] = df.groupby(GROUP_COL)[TARGET_COL].shift(lag)
        df[f"diff_{lag}"] = df[TARGET_COL] - df[f"lag_{lag}"]
    df.fillna(0, inplace=True)
    return df

# load
train_raw = add_lag_diff(add_time_feats(pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])))
val_raw   = add_lag_diff(add_time_feats(pd.read_csv(VAL_CSV, parse_dates=[TIME_COL])))
test_raw  = add_lag_diff(add_time_feats(pd.read_csv(TEST_CSV, parse_dates=[TIME_COL])))

all_candidate_feats = WEATHER_COLS + TIME_FEATS + [f"lag_{l}" for l in [1,3,6,24]] + [f"diff_{l}" for l in [1,3,6,24]]
feature_cols = [c for c in all_candidate_feats if c in train_raw.columns]
print(f"✅ feature_cols ({len(feature_cols)}): {feature_cols}")

# ----------------------------
# === 지역별 스케일러 적용 ===
# ----------------------------
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    valid_cols = [c for c in feature_cols if c in grp.columns]
    if len(valid_cols)==0: continue
    std = StandardScaler().fit(grp[valid_cols])
    mm  = MinMaxScaler().fit(std.transform(grp[valid_cols]))
    tsc = StandardScaler().fit(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))
    region_scalers[region] = (std, mm, tsc)
    train_raw.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
    train_raw.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

for df in (val_raw, test_raw):
    for region, grp in df.groupby(REGION_COL):
        valid_cols = [c for c in feature_cols if c in grp.columns]
        if region in region_scalers and len(valid_cols)>0:
            std, mm, tsc = region_scalers[region]
            df.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
            df.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

# ----------------------------
# === Dataset & Model ===
# ----------------------------
class TimeSeriesSeqDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, horizon=HORIZON, feature_cols=feature_cols):
        self.seq_len = seq_len
        self.horizon = horizon
        self.features = df[feature_cols].values.astype(np.float32)
        self.targets = df[TARGET_COL].values.astype(np.float32)
        self.n_windows = max(0, len(df) - seq_len - horizon + 1)

    def __len__(self):
        return self.n_windows

    def __getitem__(self, i):
        x = self.features[i:i + self.seq_len]
        # 단일 시점 예측이면 y도 스칼라로 반환
        if self.horizon == 1:
            y = self.targets[i + self.seq_len]
        else:
            y = self.targets[i + self.seq_len : i + self.seq_len + self.horizon]
        return torch.from_numpy(x), torch.from_numpy(np.atleast_1d(y))


class CNN_BiLSTM(nn.Module):
    def __init__(self, input_dim=len(feature_cols), hidden=128, num_layers=2, horizon=HORIZON):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 128, 3, padding=1)
        self.conv2 = nn.Conv1d(128, 64, 3, padding=1)
        self.lstm = nn.LSTM(64, hidden, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, horizon)
        self.relu = nn.ReLU()
        self.norm = nn.LayerNorm(hidden*2)
    def forward(self, x):
        x = x.permute(0,2,1)            # (B, C, T)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.permute(0,2,1)            # (B, T, C)
        x,_ = self.lstm(x)
        x = self.norm(x[:,-1,:])
        return self.fc(x)

# =========================
# === Utility Functions ===
# =========================
def calc_metrics(y_true, y_pred):
    if len(y_true)==0: return 0.0, 0.0, 0.0
    r2 = r2_score(y_true, y_pred)
    r2 = max(r2,0.0)  # 음수 R² 보정
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

def save_checkpoint(path, model, optimizer, epoch, metric, extra=None):
    ckpt = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "metric": metric
    }
    if extra: ckpt.update(extra)
    torch.save(ckpt, path)

def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt["epoch"], ckpt.get("metric",0.0)

def ema_update(prev, curr, alpha=0.3):
    if prev is None: return curr
    return alpha*curr + (1-alpha)*prev

# =========================
# === Plant 학습 함수 ===
# =========================
def train_plant(model, optimizer, criterion, train_loader, val_loader, ckpt_dir, plant):
    ckpt_last = os.path.join(ckpt_dir,f"{plant}_last.pt")
    ckpt_best = os.path.join(ckpt_dir,f"{plant}_best.pt")
    start_epoch, best_r2 = 1, -np.inf

    # Resume 기능
    if RESUME:
        if os.path.exists(ckpt_best):
            start_epoch, best_r2 = load_checkpoint(model, optimizer, ckpt_best)
            print(f"🔄 Resume from best ckpt: {ckpt_best} | start_epoch={start_epoch} | best_r2={best_r2:.4f}")
        elif os.path.exists(ckpt_last):
            start_epoch, best_r2 = load_checkpoint(model, optimizer, ckpt_last)
            print(f"🔄 Resume from last ckpt: {ckpt_last} | start_epoch={start_epoch} | best_r2={best_r2:.4f}")

    r2_ema, early_counter = None, 0
    epoch_start_time = time.time()
    for epoch in range(start_epoch, EPOCHS+1):
        # Train
        model.train()
        train_losses = []
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        train_loss_mean = np.mean(train_losses) if train_losses else float("nan")

        # Validation
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
                yp = model(Xv)
                val_preds.append(yp[:,0].cpu().numpy())
                val_true.append(yv[:,0].cpu().numpy())
                
        val_preds = np.concatenate(val_preds) if val_preds else np.array([])
        val_true = np.concatenate(val_true) if val_true else np.array([])

        # ✅ 정합성 보정
        if len(val_preds) != len(val_true):
            min_len = min(len(val_preds), len(val_true))
            val_preds, val_true = val_preds[:min_len], val_true[:min_len]

        val_r2, val_rmse, val_mae = calc_metrics(val_true, val_preds)
        r2_ema = ema_update(r2_ema, val_r2)

        best_mark=""
        if r2_ema > best_r2:
            best_r2 = r2_ema
            save_checkpoint(ckpt_best, model, optimizer, epoch, best_r2)
            best_mark="★Best★"
            early_counter=0
        else:
            early_counter+=1

        save_checkpoint(ckpt_last, model, optimizer, epoch, best_r2)

        # ETA
        avg_epoch_time = (time.time()-epoch_start_time)/max(1,epoch-start_epoch+1)
        eta_s = avg_epoch_time*(EPOCHS-epoch)
        eta_str=f"{eta_s:.1f}s" if eta_s<60 else f"{eta_s/60:.2f}m"
        print(f"Epoch {epoch}/{EPOCHS} | TrainLoss={train_loss_mean:.6f} | Val R2={val_r2:.4f} | EMA R2={r2_ema:.4f} {best_mark} | RMSE={val_rmse:.4f} | MAE={val_mae:.4f} | ETA={eta_str}")

        if early_counter>=PATIENCE:
            print(f"⏹ EarlyStopping Triggered for {plant}")
            break

    # 최종 평가
    ckpt_eval = ckpt_best if os.path.exists(ckpt_best) else ckpt_last
    ck = torch.load(ckpt_eval,map_location=DEVICE)
    model.load_state_dict(ck["model_state"])
    val_preds_list, val_true_list = [], []
    with torch.no_grad():
        for Xv, yv in val_loader:
            Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
            yp = model(Xv)
            val_preds_list.append(yp[:,0].cpu().numpy())
            val_true_list.append(yv[:,0].cpu().numpy())
    val_preds_final = np.concatenate(val_preds_list)
    val_true_final = np.concatenate(val_true_list)

    # ✅ 정합성 보정
    if len(val_preds_final) != len(val_true_final):
        min_len = min(len(val_preds_final), len(val_true_final))
        val_preds_final, val_true_final = val_preds_final[:min_len], val_true_final[:min_len]

    v_r2,v_rmse,v_mae = calc_metrics(val_true_final,val_preds_final)
    return ckpt_eval, (v_r2,v_rmse,v_mae)

# =========================
# === Plant 학습 + 2차 Weight Step ===
# =========================
results_plant_1st, results_plant_2nd = {}, {}
plant_models_1st, plant_models_2nd = {}, {}
plant_summary_rows = []  # HTML 테이블용

for plant, df_train in train_raw.groupby(GROUP_COL):
    print(f"\n🌱 Plant 1차 학습 시작: {plant} | train size: {len(df_train)}")
    df_val = val_raw[val_raw[GROUP_COL]==plant]
    if len(df_val)==0: continue

    train_loader = DataLoader(TimeSeriesSeqDataset(df_train), batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(TimeSeriesSeqDataset(df_val), batch_size=BATCH, shuffle=False)
    model = CNN_BiLSTM().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    plant_dir = os.path.join(CKPT_DIR,"plant",plant)
    os.makedirs(plant_dir, exist_ok=True)

    # 1차 학습
    ckpt_1st, metrics_1st = train_plant(model, optimizer, criterion, train_loader, val_loader, plant_dir, plant)
    plant_models_1st[plant] = ckpt_1st
    results_plant_1st[plant] = metrics_1st
    print(f"✅ Plant 1차 평가 완료 | R2={metrics_1st[0]:.4f} | RMSE={metrics_1st[1]:.4f} | MAE={metrics_1st[2]:.4f}")

    # =========================
    # === Plant별 summary row 추가
    # =========================
    r2, rmse, mae = results_plant_1st[plant]
    plant_summary_rows.append({
        "plant": plant,
        "n_train": len(df_train),
        "r2": r2,
        "rmse": rmse,
        "mae": mae
    })


# =========================
# === (NEW) 1차 전체 평가 ===
# =========================
weights_1st = np.array([len(train_raw[train_raw[GROUP_COL]==p]) for p in results_plant_1st.keys()])
r2_1st_all = np.array([results_plant_1st[p][0] for p in results_plant_1st.keys()])
rmse_1st_all = np.array([results_plant_1st[p][1] for p in results_plant_1st.keys()])
mae_1st_all = np.array([results_plant_1st[p][2] for p in results_plant_1st.keys()])
weighted_r2_1st = (r2_1st_all * weights_1st).sum() / weights_1st.sum()
weighted_rmse_1st = (rmse_1st_all * weights_1st).sum() / weights_1st.sum()
weighted_mae_1st = (mae_1st_all * weights_1st).sum() / weights_1st.sum()
print(f"\n📊 [1차 전체평가] 가중평균 R²={weighted_r2_1st:.4f} | RMSE={weighted_rmse_1st:.4f} | MAE={weighted_mae_1st:.4f}")

# 발전소 → 지역 매핑
plant_to_region = train_raw.set_index(GROUP_COL)[REGION_COL].to_dict()
# =========================
# === Region Fine-tuning ===
# =========================
region_models, region_metrics = {}, {}
region_summary_rows = []
region_to_plants = []

print("\n🌿 Region Fine-tuning 시작")

# Region별 Fine-tuning
for region, df_region in train_raw.groupby(REGION_COL):
    df_val_region = val_raw[val_raw[REGION_COL]==region]
    if len(df_val_region)==0: continue

    region_dir = os.path.join(CKPT_DIR,"region",region)
    os.makedirs(region_dir, exist_ok=True)

    plants_in_region = df_region[GROUP_COL].unique().tolist()
    plant_ckpts = [plant_models_2nd[p] for p in plants_in_region if p in plant_models_2nd]
    model = CNN_BiLSTM().to(DEVICE)

    # Plant ckpt 평균으로 초기화
    if plant_ckpts:
        ck_states = [torch.load(ck,map_location=DEVICE)["model_state"] for ck in plant_ckpts]
        avg_state = {k: torch.stack([s[k] for s in ck_states],0).mean(0) for k in ck_states[0].keys()}
        model.load_state_dict(avg_state)
        print(f"🔹 Region {region} 초기화 완료 (Plant 평균 ckpt)")

    optimizer = optim.Adam(model.parameters(), lr=LR*0.5)
    criterion = nn.MSELoss()
    train_loader = DataLoader(TimeSeriesSeqDataset(df_region), batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(TimeSeriesSeqDataset(df_val_region), batch_size=BATCH, shuffle=False)

    best_r2 = -np.inf
    best_ckpt = os.path.join(region_dir,f"{region}_best.pt")
    start_epoch = 1

    # Resume 체크
    if RESUME and os.path.exists(best_ckpt):
        start_epoch, best_r2 = load_checkpoint(model, optimizer, best_ckpt)
        print(f"🔄 Resume Region ckpt: {region} | start_epoch={start_epoch} | best_r2={best_r2:.4f}")

    for epoch in range(start_epoch, FINE_TUNE_EPOCHS+1):
        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
                yp = model(Xv)
                val_preds.append(yp[:,0].cpu().numpy())
                val_true.append(yv[:,0].cpu().numpy())
        val_preds_final = np.concatenate(val_preds)
        val_true_final = np.concatenate(val_true)

        # ✅ 정합성 보정
        if len(val_preds_final) != len(val_true_final):
            min_len = min(len(val_preds_final), len(val_true_final))
            val_preds_final, val_true_final = val_preds_final[:min_len], val_true_final[:min_len]

        r2, rmse, mae = calc_metrics(val_true_final, val_preds_final)

        if r2 > best_r2:
            best_r2 = r2
            save_checkpoint(best_ckpt, model, optimizer, epoch, best_r2)
    region_models[region] = best_ckpt
    region_metrics[region] = (best_r2, rmse, mae)
    print(f"✅ Region {region} 최종 평가 | R2={best_r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

    # =========================
    # === Region summary row 추가
    # =========================
    r2, rmse, mae = region_metrics[region]
    region_summary_rows.append({
        "region": region,
        "n_train": len(df_region),
        "r2": r2,
        "rmse": rmse,
        "mae": mae
})

# =========================
# === (NEW) 2차 전체평가 ===
# =========================
weights_2nd = np.array([len(train_raw[train_raw[REGION_COL]==p]) for p in region_metrics.keys()])
r2_2nd_all = np.array([region_metrics[p][0] for p in region_metrics.keys()])
rmse_2nd_all = np.array([region_metrics[p][1] for p in region_metrics.keys()])
mae_2nd_all = np.array([region_metrics[p][2] for p in region_metrics.keys()])
weighted_r2_2nd = (r2_2nd_all * weights_2nd).sum() / weights_2nd.sum()
weighted_rmse_2nd = (rmse_2nd_all * weights_2nd).sum() / weights_2nd.sum()
weighted_mae_2nd = (mae_2nd_all * weights_2nd).sum() / weights_2nd.sum()
print(f"\n📊 [2차 전체평가] 가중평균 R²={weighted_r2_2nd:.4f} | RMSE={weighted_rmse_2nd:.4f} | MAE={weighted_mae_2nd:.4f}")

region_to_plants = (
    train_raw.groupby(REGION_COL)[GROUP_COL]
    .unique()
    .apply(list)
    .to_dict()
)

# =========================
# === 결과 저장용 dict
# =========================
region_perm_images = {}
outlier_reports = {}
actual_vs_pred_images = {}

# =========================
# === Helper: 그래프 생성
# =========================
def plot_actual_vs_pred(region, y_true, y_pred, dates, residuals, outliers_mask):
    """이상치 탐지 후 실제값/예측값 그래프를 base64로 반환"""
    try:
        if len(dates) == 0:
            return None

        # 배열 정리
        min_len = min(len(dates), len(y_true), len(y_pred), len(residuals), len(outliers_mask))
        dates = pd.to_datetime(dates[:min_len])
        y_true = np.asarray(y_true[:min_len], dtype=float)
        y_pred = np.asarray(y_pred[:min_len], dtype=float)
        residuals = np.asarray(residuals[:min_len], dtype=float)
        outliers_mask = np.asarray(outliers_mask[:min_len], dtype=bool)

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(dates, y_true, label="실제값", alpha=0.8)
        ax.plot(dates, y_pred, label="예측값", alpha=0.8)
        if outliers_mask.sum() > 0:
            ax.scatter(
                np.array(dates)[outliers_mask],
                np.array(y_true)[outliers_mask],
                color="red",
                s=30,
                label=f"이상치 ({outliers_mask.sum()})",
                zorder=3,
            )

        # X축: 월 단위 포맷
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        plt.xticks(rotation=45)

        ax.set_title(f"{region} | Actual vs Predicted (이상치 표시)")
        ax.set_xlabel("날짜")
        ax.set_ylabel("값")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        # base64 변환
        buf = io.BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode("utf-8")
        buf.close()
        return img_b64
    except Exception as e:
        print(f"⚠ 그래프 생성 실패: {region} ({e})")
        return None

# =========================
# === 1️⃣ Permutation Importance (Train 데이터)
# =========================
def analyze_permutation_importance(region, ckpt_path, train_df):
    try:
        # 모델 로드
        ck = torch.load(ckpt_path, map_location=DEVICE)
        st = ck.get("model_state", ck)
        model.load_state_dict(st)
        model.to(DEVICE)
        model.eval()

        df_region = train_df[train_df[REGION_COL]==region].reset_index(drop=True)
        if len(df_region) == 0:
            print(f"⚠ {region}: train 데이터 없음")
            region_perm_images[region] = None
            outlier_reports.setdefault(region, {})['top3_features'] = []
            return

        ds_train = TimeSeriesSeqDataset(df_region)
        n_windows = len(ds_train)
        if n_windows == 0:
            print(f"⚠ {region}: train 윈도우 없음")
            region_perm_images[region] = None
            outlier_reports.setdefault(region, {})['top3_features'] = []
            return

        sample_size = min(PERM_SAMPLE, n_windows)
        X_sample = torch.stack([ds_train[i][0] for i in range(sample_size)]).to(DEVICE)
        y_raw = torch.stack([ds_train[i][1] for i in range(sample_size)]).cpu().numpy()
        with torch.no_grad():
            y_pred_all = model(X_sample).detach().cpu().numpy()
        y_true_col = (y_raw[:,0] if y_raw.ndim>1 else y_raw).astype(float)
        y_pred_col = y_pred_all[:,0].astype(float)
        min_len = min(len(y_true_col), len(y_pred_col))
        y_true_pi = y_true_col[:min_len]
        y_pred_pi = y_pred_col[:min_len]
        base_rmse = np.sqrt(mean_squared_error(y_true_pi, y_pred_pi))

        # Permutation Importance (weather features)
        X_np = X_sample.detach().cpu().numpy().astype(np.float32)
        seq_len = X_np.shape[1]
        weather_idx = [i for i,f in enumerate(feature_cols) if f in WEATHER_COLS]
        if len(weather_idx) == 0:
            print(f"⚠ {region}: WEATHER_COLS feature 없음")
            region_perm_images[region] = None
            outlier_reports.setdefault(region, {})['top3_features'] = []
            return

        pi_scores = np.zeros(len(weather_idx))
        for wi,f_idx in enumerate(weather_idx):
            X_perm = X_np.copy()
            for t in range(seq_len):
                np.random.shuffle(X_perm[:,t,f_idx])
            with torch.no_grad():
                y_perm = model(torch.tensor(X_perm, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy()
            y_perm_col = y_perm[:min_len,0]
            pi_scores[wi] = np.sqrt(mean_squared_error(y_true_pi, y_perm_col)) - base_rmse

        top_idx = np.argsort(pi_scores)[::-1][:3]
        top_features = [feature_cols[weather_idx[i]] for i in top_idx]
        outlier_reports.setdefault(region, {})['top3_features'] = top_features
        print(f"✅ Permutation Importance 완료: {region} | Top3: {', '.join(top_features)}")

        # 시각화
        plt.figure(figsize=(6,3))
        labels = [feature_cols[weather_idx[i]] for i in top_idx[::-1]]
        vals = pi_scores[top_idx][::-1]
        plt.barh(range(len(vals)), vals, height=0.6)
        plt.yticks(range(len(vals)), labels)
        plt.xlabel("RMSE 증가량")
        plt.title(f"[{region}] Top{len(vals)} Weather Feature Importance")
        buf = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        region_perm_images[region] = base64.b64encode(buf.read()).decode("utf-8")
        plt.close()
    except Exception as e:
        print(f"⚠ Permutation Importance 실패: {region} ({e})")
        region_perm_images[region] = None
        outlier_reports.setdefault(region, {})['top3_features'] = []

# =========================
# === 2️⃣ 이상치 탐지 (Test 데이터, 전체 월 반영)
# =========================
def detect_outliers(region, ckpt_path, test_df):
    try:
        # 모델 로드
        ck = torch.load(ckpt_path, map_location=DEVICE)
        st = ck.get("model_state", ck)
        model.load_state_dict(st)
        model.to(DEVICE)
        model.eval()

        # region 데이터 추출
        df_region = test_df[test_df[REGION_COL] == region].reset_index(drop=True)
        if len(df_region) == 0:
            print(f"⚠ {region}: test 데이터 없음")
            outlier_reports.setdefault(region, {}).update({"count":0, "ratio":0.0, "threshold":0.0})
            return

        ds_test = TimeSeriesSeqDataset(df_region)
        n_windows = len(ds_test)
        if n_windows == 0:
            print(f"⚠ {region}: test 윈도우 없음")
            outlier_reports.setdefault(region, {}).update({"count":0, "ratio":0.0, "threshold":0.0})
            return

        use_n = min(n_windows, PERM_SAMPLE)
        X_list, y_list = [], []
        for i in range(use_n):
            x_i, y_i = ds_test[i]
            X_list.append(np.asarray(x_i))
            y_list.append(np.asarray(y_i))
        X_np = np.stack(X_list)
        y_raw = np.stack(y_list)

        # 모델 예측
        with torch.no_grad():
            X_tensor = torch.tensor(X_np, dtype=torch.float32, device=DEVICE)
            y_pred_all = model(X_tensor).detach().cpu().numpy()
        y_pred_col = y_pred_all[:, 0].astype(float)
        y_true_col = (y_raw[:, 0] if y_raw.ndim > 1 else y_raw).astype(float)

        # 길이 맞춤
        min_len = min(len(y_true_col), len(y_pred_col))
        y_true_col = y_true_col[:min_len]
        y_pred_col = y_pred_col[:min_len]

        # ----------------------------
        # target_dates 생성 (윈도우 마지막 시점과 매칭)
        # ----------------------------
        pos_idx = np.array([i + SEQ_LEN - 1 for i in range(min_len)])
        pos_idx = pos_idx[pos_idx < len(df_region)]
        target_dates = pd.to_datetime(df_region.loc[pos_idx, TIME_COL].to_numpy())

        # ----------------------------
        # 2024-01-01 이후 필터링
        # ----------------------------
        mask = target_dates >= pd.Timestamp("2024-01-01")
        dates_filtered = target_dates[mask]
        y_true_f = y_true_col[mask]
        y_pred_f = y_pred_col[mask]

        # ----------------------------
        # 이상치 계산
        # ----------------------------
        residuals_f = np.abs(y_true_f - y_pred_f)
        threshold = residuals_f.mean() + 3 * residuals_f.std(ddof=0)
        outliers_mask = residuals_f > threshold
        outlier_count = int(outliers_mask.sum())
        outlier_ratio = float(outlier_count / len(residuals_f))

        outlier_reports.setdefault(region, {}).update({
            "count": outlier_count,
            "ratio": outlier_ratio,
            "threshold": float(threshold)
        })

        print(f"🚨 이상치 탐지 완료: {region} | 개수={outlier_count}, 비율={outlier_ratio:.2%}")

        # ----------------------------
        # 그래프 생성 (AutoDateLocator 적용)
        # ----------------------------
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(dates_filtered, y_true_f, label="실제값", alpha=0.8)
        ax.plot(dates_filtered, y_pred_f, label="예측값", alpha=0.8)
        if outliers_mask.sum() > 0:
            ax.scatter(
                dates_filtered[outliers_mask],
                y_true_f[outliers_mask],
                color="red",
                s=30,
                label=f"이상치 ({outliers_mask.sum()})",
                zorder=3
            )
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        plt.xticks(rotation=45)
        ax.set_title(f"{region} | Actual vs Predicted (이상치 표시)")
        ax.set_xlabel("날짜")
        ax.set_ylabel("합산발전량(Mwh)")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        buf = io.BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode("utf-8")
        buf.close()
        actual_vs_pred_images[region] = img_b64

    except Exception as e:
        print(f"⚠ 이상치 탐지 실패: {region} ({e})")
        outlier_reports.setdefault(region, {}).update({"count":0, "ratio":0.0, "threshold":0.0})

# =========================
# === 전체 실행
# =========================
for region, ckpt in region_models.items():
    analyze_permutation_importance(region, ckpt, train_raw)
for region, ckpt in region_models.items():
    detect_outliers(region, ckpt, test_raw)

# =========================
# === HTML 리포트 생성
# =========================
html_template = """
<html lang="ko">
<head>
<meta charset="utf-8"/>
<title>📊 발전소·지역별 성능 및 이상치 분석 리포트</title>
<style>
body { font-family: 'Pretendard', sans-serif; margin: 30px; background-color:#fafafa; color:#222; }
h1, h2 { margin-top: 40px; color:#333; }
table { border-collapse: collapse; width: 100%; margin: 20px 0; font-size: 13px; }
th, td { border: 1px solid #ccc; padding: 8px 12px; text-align: center; }
th { background-color: #d4edda; font-weight: bold; color:#111; }
tbody tr:nth-child(odd) { background-color: #fcfcfc; }
tbody tr:hover { background-color: #f5f9ff; }
.perm-img { width: 240px; border-radius: 10px; box-shadow: 0 2px 6px rgba(0,0,0,0.12); }
.small { font-size: 12px; color: #666; }
.card { background:white; padding:15px; border-radius:12px; box-shadow:0 3px 8px rgba(0,0,0,0.08); margin-top:20px; }
img { max-width:100%; height:auto; border-radius:8px; }

/* 컬럼별 폭 조절 */
.index-col { width:40px; font-weight: bold; }
.plant-col { width:180px; text-align:left; padding-left:10px; }
.region-col { width:120px; text-align:left; padding-left:10px; }
.metric-col { width:80px; }
.graph-col { width:260px; }
</style>
</head>
<body>

<h1>📈 발전소별 성능 요약</h1>
<table>
<thead>
<tr>
<th class="index-col"></th>
<th class="plant-col">발전소</th>
<th class="region-col">지역</th>
<th class="metric-col">표본 수</th>
<th class="metric-col">R²</th>
<th class="metric-col">RMSE</th>
<th class="metric-col">MAE</th>
</tr>
</thead>
<tbody>
{% for row in plant_rows %}
<tr>
<td class="index-col">{{ loop.index }}</td>
<td class="plant-col">{{ row.plant }}</td>
<td class="region-col">
    {{ plant_to_region.get(row.plant, '—') }}
</td>
<td class="metric-col">{{ row.n_train if row.n_train is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.r2) if row.r2 is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.rmse) if row.rmse is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.mae) if row.mae is defined else '—' }}</td>
</tr>
{% endfor %}
</tbody>
</table>

<h1>🌍 지역별 성능 및 이상치 분석</h1>
<table>
<thead>
<tr>
<th class="index-col"></th>
<th class="region-col">지역</th>
<th class="plant-col">발전소</th>
<th class="metric-col">표본 수</th>
<th class="metric-col">R² </th>
<th class="metric-col">RMSE</th>
<th class="metric-col">MAE</th>
<th class="metric-col">이상치 개수(비율)</th>
<th class="metric-col">Top3 영향 요인</th>
<th class="graph-col">영향도 그래프</th>
</tr>
</thead>
<tbody>
{% for row in region_rows %}
<tr>
<td class="index-col">{{ loop.index }}</td>
<td class="region-col">{{ row.region }}</td>
<td class="plant-col">
{% if region_to_plants and region_to_plants.get(row.region) %}
{{ ', '.join(region_to_plants[row.region]) }}
{% else %}
<span class="small">—</span>
{% endif %}
</td>
<td class="metric-col">{{ row.n_train if row.n_train is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.r2) if row.r2 is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.rmse) if row.rmse is defined else '—' }}</td>
<td class="metric-col">{{ "%.4f"|format(row.mae) if row.mae is defined else '—' }}</td>
<td class="metric-col">
{% if outlier_reports and outlier_reports.get(row.region) %}
{{ outlier_reports[row.region]['count'] }} ({{"%.2f"|format(outlier_reports[row.region]['ratio']*100)}}%)
{% else %} — {% endif %}
</td>
<td class="metric-col">
{% if outlier_reports and outlier_reports.get(row.region) and outlier_reports[row.region].get('top3_features') %}
{{ ', '.join(outlier_reports[row.region]['top3_features']) }}
{% else %} — {% endif %}
</td>
<td class="graph-col">
{% if region_perm_images and region_perm_images.get(row.region) %}
<img class="perm-img" src="data:image/png;base64,{{ region_perm_images[row.region] }}" alt="pi-{{row.region}}"/>
{% else %}
<span class="small">N/A</span>
{% endif %}
</td>
</tr>
{% endfor %}
</tbody>
</table>

<div class="card">
<h2>📘 전체 가중 평균 성능 요약</h2>
<p class="middle">
1차 학습 가중 평균 → R²={{"%.4f"|format(weighted_r2_1st)}} | RMSE={{"%.4f"|format(weighted_rmse_1st)}} | MAE={{"%.4f"|format(weighted_mae_1st)}}<br>
2차 학습 가중 평균 → R²={{"%.4f"|format(weighted_r2_2nd)}} | RMSE={{"%.4f"|format(weighted_rmse_2nd)}} | MAE={{"%.4f"|format(weighted_mae_2nd)}}
</p>
</div>

<h1>📊 지역별 예측 결과 그래프</h1>
{% for region, img_b64 in actual_vs_pred_images.items() %}
<div class="card">
  <h2>{{ region }}</h2>
  {% if outlier_reports and outlier_reports.get(region) %}
  <p class="small">
    이상치 개수: {{ outlier_reports[region]['count'] }} |
    비율: {{"%.2f"|format(outlier_reports[region]['ratio']*100)}}% |
    임계값: {{"%.4f"|format(outlier_reports[region]['threshold'])}}
  </p>
  {% endif %}
  <img src="data:image/png;base64,{{ img_b64 }}" alt="actual-pred-{{region}}">
</div>
{% endfor %}

</body>
</html>

"""

# =========================
# HTML 렌더링
# =========================
template = Template(html_template)
html_out = template.render(
    plant_rows=plant_summary_rows,
    region_rows=region_summary_rows,
    region_to_plants=region_to_plants,
    plant_to_region = plant_to_region,
    region_perm_images=region_perm_images,
    outlier_reports=outlier_reports,
    actual_vs_pred_images=actual_vs_pred_images,
    weighted_r2_1st=weighted_r2_1st,
    weighted_rmse_1st=weighted_rmse_1st,
    weighted_mae_1st=weighted_mae_1st,
    weighted_r2_2nd=weighted_r2_2nd,
    weighted_rmse_2nd=weighted_rmse_2nd,
    weighted_mae_2nd=weighted_mae_2nd,
)

with open("region_outlier_perm_report.html", "w", encoding="utf-8") as f:
    f.write(html_out)

html_file = os.path.join(SAVE_DIR,"plant_region_report_perm.html")
with open(html_file,"w",encoding="utf-8") as f:
    f.write(html_out)
webbrowser.open(f"file://{html_file}")
print(f"✅ HTML 보고서 생성 완료 (perm 포함): {html_file}")


✅ Device: cuda
✅ feature_cols (22): ['기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '증기압(hPa)', '일조(hr)', '일사(MJ/m2)', '적설(cm)', '전운량(10분위)', '중하층운량(10분위)', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_3', 'lag_6', 'lag_24', 'diff_1', 'diff_3', 'diff_6', 'diff_24']

🌱 Plant 1차 학습 시작: 남제주소내 | train size: 78888
🔄 Resume from best ckpt: C:\ESG_Project1\cnn_lstm\output\checkpoints\plant\남제주소내\남제주소내_best.pt | start_epoch=5 | best_r2=0.9111
Epoch 5/50 | TrainLoss=0.141350 | Val R2=0.9012 | EMA R2=0.9012  | RMSE=0.2779 | MAE=0.1578 | ETA=11.65m
⏹ EarlyStopping Triggered for 남제주소내
✅ Plant 1차 평가 완료 | R2=0.9111 | RMSE=0.2636 | MAE=0.1534

🌱 Plant 1차 학습 시작: 부산복합자재창고 | train size: 78888
🔄 Resume from best ckpt: C:\ESG_Project1\cnn_lstm\output\checkpoints\plant\부산복합자재창고\부산복합자재창고_best.pt | start_epoch=11 | best_r2=0.8709
Epoch 11/50 | TrainLoss=0.011700 | Val R2=0.8558 | EMA R2=0.8558  | RMSE=0.1287 | MAE=0.0698 | ETA=10.00m
⏹ EarlyStopping Triggered for 부산복합자재창고
✅ Plant 1차 평가 완료 | R2=0.870